In [135]:
data = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'
!wget $data

--2026-10-11 09:52:49--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 278746 (272K) [text/plain]
Saving to: ‘course_lead_scoring_2026.csv.5’

course_lead_scoring 100%[===================>] 272.21K  --.-KB/s    in 0.003s  

2026-10-11 09:52:49 (94.6 MB/s) - ‘course_lead_scoring_2026.csv.5’ saved [278746/278746]



In [136]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [137]:
df = pd.read_csv('course_lead_scoring_2026.csv')
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [138]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [139]:
df.columns
df.nunique()

lead_source                    5
industry                       6
employment_status              4
location                       5
annual_income               4313
number_of_courses_viewed       7
interaction_count             14
lead_score                    99
converted                      2
dtype: int64

In [140]:
categorical = ['lead_source', 'industry', 'employment_status', 'location']
numerical = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']

In [141]:
df.isna().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [142]:
df[categorical] = df[categorical].fillna('NA') # not including converted anyway
df[numerical] = df[numerical].fillna(0.0)
df.isna().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [143]:
# Q1 mode of industry
df.industry.mode()

0    technology
Name: industry, dtype: str

In [144]:
# Q2 correlation matrix

# interaction_count and lead_score
# number_of_courses_viewed and lead_score
# number_of_courses_viewed and interaction_count
# annual_income and interaction_count

print(df[['interaction_count']].corrwith(df.lead_score))
print(df[['number_of_courses_viewed']].corrwith(df.lead_score))
print(df[['number_of_courses_viewed']].corrwith(df.interaction_count))
print(df[['annual_income']].corrwith(df.interaction_count))

interaction_count    0.915746
dtype: float64
number_of_courses_viewed    0.757204
dtype: float64
number_of_courses_viewed    0.721609
dtype: float64
annual_income    0.122842
dtype: float64


In [145]:
# Split data
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

In [146]:
del df_train['converted']
del df_val['converted']
del df_test['converted']

In [147]:
df_train.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score
0,paid_ads,finance,employed,asia,65980.0,2,3,0.37
1,events,finance,employed,south_america,71662.0,0,0,0.09
2,referral,NA,employed,asia,40220.0,2,5,0.55
3,organic_search,finance,employed,north_america,88352.0,2,2,0.45
4,events,technology,employed,north_america,89316.0,1,5,0.00


In [148]:
y_train

array([1, 0, 1, ..., 1, 0, 0], shape=(3000,))

In [149]:
# Q3 Mutual Information
from sklearn.metrics import mutual_info_score
# industry
# location
# lead_source
# employment_status
# Question 4

score_industry = round(mutual_info_score(df_full_train.industry, df_full_train.converted), 2)
score_location = round(mutual_info_score(df_full_train.location, df_full_train.converted), 2)
score_leadsource = round(mutual_info_score(df_full_train.lead_source, df_full_train.converted), 2)
score_eimploymentstatus = round(mutual_info_score(df_full_train.employment_status, df_full_train.converted), 2)

print(score_industry, score_location, score_leadsource, score_eimploymentstatus)

0.0 0.0 0.03 0.02


In [150]:
# One-hot encoding
from sklearn.feature_extraction import DictVectorizer

dicts = df_train[categorical+numerical].iloc[:100].to_dict(orient='records')
dv = DictVectorizer(sparse=False)  # sparse = Sparse Matrix
print('\n\n')

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)
print(dv.get_feature_names_out())
print(X_train.shape)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)
print(dv.get_feature_names_out())
print(X_val.shape)




['annual_income' 'employment_status=NA' 'employment_status=employed'
 'employment_status=self_employed' 'employment_status=student'
 'employment_status=unemployed' 'industry=NA' 'industry=education'
 'industry=finance' 'industry=healthcare' 'industry=manufacturing'
 'industry=retail' 'industry=technology' 'interaction_count' 'lead_score'
 'lead_source=NA' 'lead_source=events' 'lead_source=organic_search'
 'lead_source=paid_ads' 'lead_source=referral' 'lead_source=social_media'
 'location=NA' 'location=africa' 'location=asia' 'location=europe'
 'location=north_america' 'location=south_america'
 'number_of_courses_viewed']
(3000, 28)
['annual_income' 'employment_status=NA' 'employment_status=employed'
 'employment_status=self_employed' 'employment_status=student'
 'employment_status=unemployed' 'industry=NA' 'industry=education'
 'industry=finance' 'industry=healthcare' 'industry=manufacturing'
 'industry=retail' 'industry=technology' 'interaction_count' 'lead_score'
 'lead_source=NA'

In [151]:
# Logistic Regression
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1] # [0 | 1]
converted_decision = (y_pred >= 0.5)

In [152]:
# Q4

df_pred = pd.DataFrame()
df_pred['probability'] = y_pred
df_pred['prediction'] = converted_decision.astype(int)
df_pred['actual'] = y_val

df_pred['correct'] = df_pred.prediction == df_pred.actual
df_pred['correct'].mean() # 0.655 (if included numerical: 0.645)

np.float64(0.645)

In [153]:
# Q5 Feature elimination 

all_cats = categorical + numerical
for e in ['lead_score','number_of_courses_viewed','interaction_count']:
    
    # print(f"all_cats: {all_cats}")
    cat = all_cats
    cat.remove(e)
    # print(f"cat: {cat}")
    newdf_train = df_train[cat]
    newdf_val = df_val[cat]
    
    train_dicts = newdf_train[cat].to_dict(orient='records')
    # one hot
    X_train = dv.fit_transform(train_dicts)
    # print(dv.get_feature_names_out())
    # print(X_train.shape)
    
    val_dicts = newdf_val[cat].to_dict(orient='records')
    X_val = dv.transform(val_dicts)
    # print(dv.get_feature_names_out())
    # print(X_val.shape)

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict_proba(X_val)[:, 1] # [0 | 1]
    converted_decision = (y_pred >= 0.5)

    df_pred = pd.DataFrame()
    df_pred['probability'] = y_pred
    df_pred['prediction'] = converted_decision.astype(int)
    df_pred['actual'] = y_val
    
    df_pred['correct'] = df_pred.prediction == df_pred.actual
    print(f"{e}: {df_pred['correct'].mean()}") # lead score has least difference

lead_score: 0.644
number_of_courses_viewed: 0.643
interaction_count: 0.6


In [154]:
# Q6
    
tot = categorical + numerical

train_dicts = df_train[tot].to_dict(orient='records')
# one hot
X_train = dv.fit_transform(train_dicts)
# print(dv.get_feature_names_out())
# print(X_train.shape)

val_dicts = df_val[tot].to_dict(orient='records')
X_val = dv.transform(val_dicts)
# print(dv.get_feature_names_out())
# print(X_val.shape)

for c in [0.000001, 0.00001, 0.0001, 0.001]:

    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict_proba(X_val)[:, 1] # [0 | 1]
    converted_decision = (y_pred >= 0.5)

    df_pred = pd.DataFrame()
    df_pred['probability'] = y_pred
    df_pred['prediction'] = converted_decision.astype(int)
    df_pred['actual'] = y_val
    
    df_pred['correct'] = df_pred.prediction == df_pred.actual
    print(f"{c}: {df_pred['correct'].mean()}") # c = 0.001
    

1e-06: 0.598
1e-05: 0.598
0.0001: 0.613
0.001: 0.645
